In [ ]:
"""
16-QAM Fully-Quantized ResNet on CIFAR-10
==========================================
All inputs, weights, and activations quantized to 16-QAM:
  4 levels per real component: [-3,-1,+1,+3] / √10
                              ≈ [-0.9487, -0.3162, +0.3162, +0.9487]

Target: 85%+ test accuracy (requirement: 82%+)

Key ideas borrowed from BNN/QNN literature:
  ① Pre-activation ResNet with FLOAT skip connections [Bi-Real Net, 2018]
     — Float skip = gradient highway around quantized ops.
       Without it, STE approximation compounds over 24 layers and training
       collapses. This is THE single most important change from the QPSK code.
  ② Per-output-channel learned scale α on weights [XNOR-Net, 2016]
     — Quantized levels {±0.316, ±0.949} are fixed magnitudes. α lets each
       output filter scale its effective weight magnitude independently.
       Recovers ~3-5% accuracy vs no scaling.
  ③ Per-channel learned scale α on activations
     — BN output is N(0,1). α controls how many inputs hit outer vs inner
       levels. Network learns the right tradeoff per feature map.
  ④ tanh normalization before snap (weights + activations)
     — tanh smoothly bounds any real value to the QAM range [-max, +max].
       For weights: avoids hard gradient cutoff that clipping would cause.
       For activations: maps N(0,1) BN output to QAM range without outliers
       dominating the outermost levels.
  ⑤ Clipped STE [PACT, 2018]
     — Only passes gradient where |x| ≤ 1.1 × max_level.
       Values deep outside the constellation always snap to the outermost
       level regardless of gradient sign — clipping avoids misleading updates.
  ⑥ CutMix augmentation [Yun et al., 2019] — adds ~2% on CIFAR-10.
  ⑦ AutoAugment (CIFAR10 policy) [Cubuk et al., 2019] — adds ~1%.
  ⑧ Label smoothing — prevents overconfident logits, better generalization.
  ⑨ Cosine LR with linear warmup — stable for quantized networks.

Architecture: QAM16ResNet (base_ch=64, n_blocks=4 per stage)
  QAM16 input → Float stem (3→64) → 3 × [4× QResBlock] → Float head → 10 logits
  Channels:  64 →  64 (32×32) → 128 (16×16) → 256 (8×8) → GlobalAvgPool
  Float params: stem + skip projections + head ≈ 3% of total
  Quant params: all main-path convs ≈ 97% of total

Expected accuracy: 85–88% (safely above 82% target)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import math
import numpy as np

# ════════════════════════════════════════════════════════════════════════════
# 0.  DEVICE
# ════════════════════════════════════════════════════════════════════════════
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ════════════════════════════════════════════════════════════════════════════
# 1.  16-QAM CONSTELLATION CONSTANTS
# ════════════════════════════════════════════════════════════════════════════
# 16-QAM: 4×4 grid in 2D complex plane.
# Each real axis: levels [-3,-1,+1,+3].
# Normalize so total symbol power E[|s|²] = 1:
#   E[sr²] = E[si²] = (9+1+1+9)/4 = 5  →  E[|s|²] = 10  →  normalize by √10
# Per-component power = 0.5, total = 1.
_QAM16_RAW  = torch.tensor([-3., -1., 1., 3.])
_QAM16_NORM = math.sqrt(10.0)                            # = √10 ≈ 3.162
QAM16_LEVELS = _QAM16_RAW / _QAM16_NORM                 # ≈ [-0.9487,-0.3162,+0.3162,+0.9487]
QAM16_MAX    = float(QAM16_LEVELS[-1])                   # ≈ 0.9487

# ════════════════════════════════════════════════════════════════════════════
# 2.  CORE STE QUANTIZER — Argmin Snap + Clipped Straight-Through Estimator
# ════════════════════════════════════════════════════════════════════════════
class _QAM16SnapFunc(torch.autograd.Function):
    """
    Forward : snap each element to the nearest of the 4 QAM levels (argmin).
    Backward: clipped STE — gradient passes unchanged where |x| ≤ 1.1 × QAM16_MAX,
              zero elsewhere. This is strictly better than vanilla STE:
              deeply-saturated values always snap to the outermost level
              regardless of gradient direction — clipping avoids spurious updates.

    Works correctly for ANY tensor shape (1D activations, 4D conv features,
    4D weight tensors). Uses x.dim() to build the right broadcast shape.

    Bug note: The previous QPSK code used step-based rounding (x/step → round → ×step)
    which accidentally collapses ALL outputs to zero for QPSK (step=2.0, tanh output
    ∈ [-1,1] → scaled to [-0.5,0.5] → all round to 0). Argmin snap is correct for
    all QAM orders and is used throughout this codebase.
    """
    @staticmethod
    def forward(ctx, x: torch.Tensor, levels: torch.Tensor) -> torch.Tensor:
        ctx.save_for_backward(x)
        # Reshape levels to (...,4) for broadcasting: e.g. (1,1,1,1,4) for 4D x
        lv    = levels.view(*([1] * x.dim()), -1)    # (..., 4)
        dists = (x.unsqueeze(-1) - lv).abs()          # (..., 4)
        idx   = dists.argmin(dim=-1)                  # (...)  — index of nearest level
        return levels[idx]                            # (...)  — quantized output

    @staticmethod
    def backward(ctx, grad: torch.Tensor):
        (x,) = ctx.saved_tensors
        # Clipped STE: pass gradient only within ±(1.1 × max constellation level)
        mask = (x.abs() <= QAM16_MAX * 1.1).to(grad.dtype)
        return grad * mask, None    # None: levels buffer needs no gradient


def qam16_snap(x: torch.Tensor, levels: torch.Tensor) -> torch.Tensor:
    """Public wrapper: argmin snap to 4 QAM levels with clipped STE."""
    return _QAM16SnapFunc.apply(x, levels)


# ════════════════════════════════════════════════════════════════════════════
# 3.  INPUT QUANTIZER  (no learned parameters)
# ════════════════════════════════════════════════════════════════════════════
class QAM16InputQ(nn.Module):
    """
    Hard 16-QAM quantization of normalized RGB pixels. Zero learnable params.

    Pipeline:
        normalized pixel  →  tanh(x) × QAM16_MAX  →  argmin snap → QAM level

    Why tanh?
      CIFAR-10 normalized pixels ≈ [-2.5, +2.5].
      QAM levels span [-0.949, +0.949].
      Without bounding: almost all pixels snap to ±0.949 (outermost levels only),
      wasting the inner levels entirely.
      tanh smoothly maps ℝ → [-1,+1], scaled to [-0.949,+0.949], so all 4
      levels get meaningful usage and the quantizer has full gradient flow.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer('levels', QAM16_LEVELS.clone())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return qam16_snap(torch.tanh(x) * QAM16_MAX, self.levels)

    @torch.no_grad()
    def verify(self, x: torch.Tensor) -> None:
        """Sanity check: must see exactly 4 distinct output levels."""
        xq = self.forward(x)
        u  = xq.unique()
        ok = "✓  OK" if u.numel() == 4 else "✗  BUG — expected 4 unique levels!"
        print(f"  Unique levels : {u.numel()}  {ok}")
        print(f"  Level values  : {[f'{v:.4f}' for v in u.tolist()]}")
        print(f"  Output range  : [{xq.min():.4f}, {xq.max():.4f}]")


# ════════════════════════════════════════════════════════════════════════════
# 4.  ACTIVATION QUANTIZER  (per-channel learned scale)
# ════════════════════════════════════════════════════════════════════════════
class QAM16ActQ(nn.Module):
    """
    Quantizes conv feature-map activations to 4 QAM levels.
    Includes a learnable per-channel scale α.

    Pipeline:
        BN output  →  tanh(x × |α|) × QAM16_MAX  →  argmin snap

    Why learned α?
      After BN, activations ~ N(0,1). With fixed α=1:
        tanh(±1) ≈ ±0.76 → maps to inner levels ±0.316 for ~68% of values.
      Some feature maps benefit from more outer-level usage (large α → sharper
      tanh → more values hit ±0.949) or more inner-level usage (small α).
      Learned α lets each channel self-tune.

    α is clamped to [0.1, 5.0] for numerical stability.
    Shape: (1, C, 1, 1) to broadcast over (B, C, H, W).
    """
    def __init__(self, num_channels: int):
        super().__init__()
        self.register_buffer('levels', QAM16_LEVELS.clone())
        self.alpha = nn.Parameter(torch.ones(1, num_channels, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = self.alpha.abs().clamp(min=0.1, max=5.0)
        return qam16_snap(torch.tanh(x * alpha) * QAM16_MAX, self.levels)


# ════════════════════════════════════════════════════════════════════════════
# 5.  QUANTIZED CONV2D  (16-QAM weights + per-output-channel scale)
# ════════════════════════════════════════════════════════════════════════════
class QAM16Conv2d(nn.Module):
    """
    Conv2d with 16-QAM quantized weights and a per-output-channel scale α.

    Weight pipeline:
        w_float  →  tanh(w) × QAM16_MAX  →  argmin snap  →  × |α_c|  →  w_final

    Step 1 — tanh normalization:
      Maps unconstrained float weights to [-0.949, +0.949] = QAM constellation range.
      Unlike hard clipping, tanh always has nonzero gradient (sech²(w) > 0 for all w),
      so even large latent weights keep receiving meaningful gradient updates.

    Step 2 — argmin snap (STE):
      Hard-quantizes to one of the 4 QAM levels. Clipped STE in backward.

    Step 3 — per-channel scale α_c:
      After snapping, quantized values are locked to {±0.316, ±0.949}.
      Different output filters may need very different effective magnitudes.
      α_c (one scalar per output channel) multiplies the quantized filter,
      restoring the magnitude degree of freedom without adding many params.
      Initialized to 0.5 (moderate initial effective weight magnitude).
      α_c is clamped to [1e-3, 10.0].
    """
    def __init__(self, in_ch: int, out_ch: int, kernel: int,
                 stride: int = 1, padding: int = 0):
        super().__init__()
        self.stride  = stride
        self.padding = padding
        self.weight  = nn.Parameter(torch.empty(out_ch, in_ch, kernel, kernel))
        # Per-output-channel scale: shape (out_ch, 1, 1, 1) broadcasts over weight
        self.alpha   = nn.Parameter(torch.full((out_ch, 1, 1, 1), 0.5))
        self.register_buffer('levels', QAM16_LEVELS.clone())
        nn.init.kaiming_normal_(self.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w_norm  = torch.tanh(self.weight) * QAM16_MAX           # → [-0.949, +0.949]
        w_q     = qam16_snap(w_norm, self.levels)                # → {-0.949,-0.316,+0.316,+0.949}
        w_final = w_q * self.alpha.abs().clamp(min=1e-3, max=10.0)
        return F.conv2d(x, w_final, stride=self.stride, padding=self.padding)


# ════════════════════════════════════════════════════════════════════════════
# 6.  QUANTIZED RESIDUAL BLOCK  (pre-activation, float skip)
# ════════════════════════════════════════════════════════════════════════════
class QResBlock(nn.Module):
    """
    Pre-activation 16-QAM residual block.

    Main path (all quantized):
        BN₁ → QAM16_act₁ → QConv₁(3×3) → BN₂ → QAM16_act₂ → QConv₂(3×3)

    Skip (FLOAT — the gradient highway):
        • Same channels & stride → nn.Identity()
        • Otherwise → AvgPool(stride) + float Conv2d(1×1) + BN

    Output: main + skip   (float element-wise addition)

    ──────────────────────────────────────────────────────────────────────
    WHY FLOAT SKIP CONNECTIONS ARE ESSENTIAL FOR QUANTIZED NETWORKS
    ──────────────────────────────────────────────────────────────────────
    In a standard (float) network, gradients flow cleanly through all layers.
    In a quantized network, every QAM16 snap introduces STE approximation error.
    Over 24 quantized conv layers (12 blocks × 2 convs), these errors compound:
    the gradient seen by early layers is a heavily-distorted version of the
    true loss gradient. Training fails — this is exactly what caused the QPSK
    code to stall at ~10% (combined with the zero-output bug).

    Float skip connections provide a parallel path with EXACT gradient flow.
    The loss gradient can travel back through the skip connections without
    ANY quantization error. The quantized main path gets to see gradients
    that are a mix of exact (from skip) and approximate (from main), which
    is close enough to train well.

    This insight is from "Bi-Real Net" (Liu et al., 2018) and is now the
    standard design pattern for all competitive quantized networks.

    ──────────────────────────────────────────────────────────────────────
    WHY PRE-ACTIVATION ORDER (BN → Q → Conv)?
    ──────────────────────────────────────────────────────────────────────
    BN normalizes activations to approximately N(0,1) BEFORE quantization.
    This gives the quantizer a well-conditioned input:
      • tanh(N(0,1)) maps to roughly [-0.76, +0.76]
      • All 4 QAM levels get meaningful usage
    If we quantized AFTER conv (post-activation), the scale is arbitrary
    and most values might crowd the outermost levels, wasting inner levels.
    """
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()

        # ── Main path (quantized) ─────────────────────────────────────────
        self.bn1    = nn.BatchNorm2d(in_ch)
        self.qact1  = QAM16ActQ(in_ch)
        self.conv1  = QAM16Conv2d(in_ch, out_ch, 3, stride=stride, padding=1)

        self.bn2    = nn.BatchNorm2d(out_ch)
        self.qact2  = QAM16ActQ(out_ch)
        self.conv2  = QAM16Conv2d(out_ch, out_ch, 3, stride=1, padding=1)

        # ── Skip connection (float) ───────────────────────────────────────
        if stride != 1 or in_ch != out_ch:
            layers = []
            if stride > 1:
                # AvgPool for spatial downsampling (better than stride-conv for skips:
                # no aliasing, and the float skip stays truly float with no weights)
                layers.append(nn.AvgPool2d(kernel_size=stride, stride=stride))
            layers += [
                nn.Conv2d(in_ch, out_ch, 1, bias=False),  # float 1×1 channel projection
                nn.BatchNorm2d(out_ch)
            ]
            self.skip = nn.Sequential(*layers)
        else:
            self.skip = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Float skip: computed first, provides exact gradient path in backward
        identity = self.skip(x)

        # Quantized main path: BN → Q_act → QConv → BN → Q_act → QConv
        out = self.conv1(self.qact1(self.bn1(x)))
        out = self.conv2(self.qact2(self.bn2(out)))

        # Float accumulation: gradient splits between skip (exact) and main (STE-approx)
        return out + identity


# ════════════════════════════════════════════════════════════════════════════
# 7.  FULL NETWORK: QAM16ResNet
# ════════════════════════════════════════════════════════════════════════════
class QAM16ResNet(nn.Module):
    """
    16-QAM Quantized ResNet for CIFAR-10.

    Layer overview (base_ch=64, n_blocks=4):
    ┌─────────────┬───────────────┬────────────┬──────────┬────────────┐
    │  Component  │  Type         │ Channels   │ Spatial  │ Float?     │
    ├─────────────┼───────────────┼────────────┼──────────┼────────────┤
    │ input_q     │ QAM16InputQ   │ 3          │ 32×32    │ no (STE)   │
    │ stem        │ Conv + BN     │ 3 → 64     │ 32×32    │ YES        │
    │ stage1      │ 4× QResBlock  │ 64 → 64    │ 32×32    │ skip only  │
    │ stage2      │ 4× QResBlock  │ 64 → 128   │ 16×16    │ skip only  │
    │ stage3      │ 4× QResBlock  │ 128 → 256  │ 8×8      │ skip only  │
    │ head        │ BN+GAP+FC     │ 256 → 10   │ 1×1      │ YES        │
    └─────────────┴───────────────┴────────────┴──────────┴────────────┘

    Float params: stem (~1.7K) + 2 skip projections (~41K) + head (~2.6K)
    Quantized:    all 24 main-path conv layers (~4.7M params)
    Float ratio:  ~1% of total parameters
    """
    def __init__(self, num_classes: int = 10,
                 base_ch: int = 64, n_blocks: int = 4):
        super().__init__()
        c1, c2, c3 = base_ch, base_ch * 2, base_ch * 4

        # 16-QAM input quantizer (no learned params)
        self.input_q = QAM16InputQ()

        # Float first conv: only 3 input channels, tiny parameter cost,
        # but allows the network to form rich initial feature representations
        # before quantized residual processing begins.
        self.stem = nn.Sequential(
            nn.Conv2d(3, base_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch)
        )

        # Three quantized residual stages
        self.stage1 = self._make_stage(base_ch, c1, n_blocks, stride=1)
        self.stage2 = self._make_stage(c1,      c2, n_blocks, stride=2)
        self.stage3 = self._make_stage(c2,      c3, n_blocks, stride=2)

        # Float classification head
        self.head = nn.Sequential(
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=0.1),         # light dropout — quantization already regularizes
            nn.Linear(c3, num_classes)
        )

        self._init_weights()

    @staticmethod
    def _make_stage(ic: int, oc: int, n: int, stride: int) -> nn.Sequential:
        """First block handles channel/spatial change; rest maintain dimensions."""
        return nn.Sequential(
            QResBlock(ic, oc, stride),
            *[QResBlock(oc, oc, stride=1) for _ in range(n - 1)]
        )

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_q(x)       # 16-QAM input quantization (STE in backward)
        x = self.stem(x)           # float: extract 64 initial feature maps, 32×32
        x = self.stage1(x)         # quantized: 64ch, 32×32
        x = self.stage2(x)         # quantized: 128ch, 16×16  (stride-2 downsample)
        x = self.stage3(x)         # quantized: 256ch,  8×8   (stride-2 downsample)
        return self.head(x)        # float: 256 → 10 class logits

    def param_summary(self) -> tuple:
        """(total_params, float_params, quant_params)"""
        total = sum(p.numel() for p in self.parameters())
        float_p = sum(p.numel() for p in self.stem.parameters())
        float_p += sum(p.numel() for p in self.head.parameters())
        for m in self.modules():
            if isinstance(m, QResBlock) and not isinstance(m.skip, nn.Identity):
                float_p += sum(p.numel() for p in m.skip.parameters())
        return total, float_p, total - float_p


# ════════════════════════════════════════════════════════════════════════════
# 8.  DATA LOADING
# ════════════════════════════════════════════════════════════════════════════
def get_dataloaders(batch_size: int = 128, num_workers: int = 4):
    """
    CIFAR-10 loaders with state-of-the-art augmentation stack.
    Train: AutoAugment(CIFAR10) + RandomErasing (+ CutMix applied in training loop)
    Test : normalize only
    """
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)

    try:
        aa = transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10)
        train_tf = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            aa,
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
        ])
        print("  Augmentation : AutoAugment(CIFAR10) + RandomErasing + CutMix")
    except AttributeError:
        # torchvision < 0.12 fallback
        train_tf = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        print("  Augmentation : Basic (upgrade torchvision for AutoAugment) + CutMix")

    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    nw = 0 if device.type == 'mps' else num_workers
    kw = dict(num_workers=nw,
              pin_memory=(device.type == 'cuda'),
              persistent_workers=(nw > 0))

    train_set = torchvision.datasets.CIFAR10('./data', True,  train_tf, download=True)
    test_set  = torchvision.datasets.CIFAR10('./data', False, test_tf,  download=True)

    return (
        torch.utils.data.DataLoader(train_set, batch_size, shuffle=True,  **kw),
        torch.utils.data.DataLoader(test_set,  batch_size, shuffle=False, **kw)
    )


# ════════════════════════════════════════════════════════════════════════════
# 9.  TRAINING UTILITIES
# ════════════════════════════════════════════════════════════════════════════
def cutmix(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    """
    CutMix augmentation (Yun et al., ICCV 2019).
    Pastes a rectangle from one image onto another; mixes labels by area ratio.
    Returns: mixed_images, labels_a, labels_b, lambda_actual
    """
    lam  = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape

    # Box dimensions: area ≈ (1 - lam) of image
    ch = int(H * math.sqrt(1.0 - lam))
    cw = int(W * math.sqrt(1.0 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = max(0, cy - ch // 2), min(H, cy + ch // 2)
    x1, x2 = max(0, cx - cw // 2), min(W, cx + cw // 2)

    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (y2 - y1) * (x2 - x1) / (H * W)

    return x_mix, y, y[perm], lam_actual


def smooth_ce(outputs: torch.Tensor,
              ya: torch.Tensor,
              yb: torch.Tensor = None,
              lam: float = 1.0,
              smoothing: float = 0.1,
              nc: int = 10) -> torch.Tensor:
    """
    Label-smoothed cross-entropy with optional CutMix label mixing.

    Label smoothing converts hard one-hot targets to soft targets:
      correct class:   1 - ε          (default: 0.9)
      other classes:   ε / (K-1)      (default: 0.0111 each)
    Prevents overconfident predictions, improves calibration and generalization.
    """
    log_p = F.log_softmax(outputs, dim=1)

    def nll(targets: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            soft = torch.full_like(log_p, smoothing / (nc - 1))
            soft.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
        return -(soft * log_p).sum(dim=1).mean()

    if yb is None:
        return nll(ya)
    return lam * nll(ya) + (1.0 - lam) * nll(yb)


def make_scheduler(optimizer, warmup_epochs: int, total_epochs: int):
    """Linear LR warmup (epochs 1..warmup_epochs) then cosine annealing to 0."""
    def lr_fn(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs          # 0 → 1 linearly
        t = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * t))     # 1 → ~0 via cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


# ════════════════════════════════════════════════════════════════════════════
# 10.  TRAINING AND EVALUATION LOOPS
# ════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer,
                epoch: int, warmup: int, cutmix_p: float,
                log_grads: bool = False):
    model.train()
    tot_loss = correct = total = 0

    for bidx, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)

        # CutMix: activate after warmup with probability cutmix_p
        # During warmup, plain training is more stable for quantized networks
        if epoch > warmup and np.random.random() < cutmix_p:
            imgs, la, lb, lam = cutmix(imgs, labels)
            out  = model(imgs)
            loss = smooth_ce(out, la, lb, lam)
        else:
            out  = model(imgs)
            loss = smooth_ce(out, labels)

        optimizer.zero_grad()
        loss.backward()

        # ── Gradient check (epoch 1, batch 0 only) ─────────────────────
        if bidx == 0 and log_grads:
            print("\n  [GRADIENT CHECK — Epoch 1, Batch 0]")
            all_dead = True
            for name, p in model.named_parameters():
                if p.grad is not None and p.grad.abs().mean() > 1e-9:
                    all_dead = False
                    if 'weight' in name:
                        g = p.grad.abs().mean().item()
                        print(f"  {name:50s} grad={g:.2e}  ✓")
            if all_dead:
                print("  ✗  ALL GRADIENTS ZERO — backward pass broken!")
            else:
                print("  ✓  Gradients flowing correctly.")
            print()

        # Gradient clipping: quantized-net gradients can be noisy
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        tot_loss += loss.item()
        _, pred = out.max(1)
        correct += pred.eq(labels).sum().item()
        total   += labels.size(0)

    return tot_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader) -> float:
    model.eval()
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        _, pred = model(imgs).max(1)
        correct += pred.eq(labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total


# ════════════════════════════════════════════════════════════════════════════
# 11.  MAIN
# ════════════════════════════════════════════════════════════════════════════
def main():
    # ── Hyperparameters ──────────────────────────────────────────────────────
    EPOCHS       = 200      # 200 epochs with cosine LR converges well on CIFAR-10
    BATCH_SIZE   = 128      # safe for 8GB+ VRAM; increase to 256 if memory allows
    LR           = 3e-3     # peak AdamW LR (reached after warmup)
    WEIGHT_DECAY = 1e-4     # applied to float/weight params; not to BN/bias/scale
    WARMUP       = 15       # linear LR warmup epochs (stabilizes early quantized training)
    CUTMIX_PROB  = 0.5      # probability of CutMix per batch (after warmup)
    BASE_CH      = 64       # channel progression: 64 → 128 → 256
    N_BLOCKS     = 4        # residual blocks per stage (12 total = 24 quantized convs)
    DIAG_EVERY   = 25       # print per-layer diagnostics every N epochs
    SAVE_NAME    = "qam16_resnet_cifar10_best.pth"

    print("=" * 72)
    print("  16-QAM FULLY-QUANTIZED ResNet — CIFAR-10")
    print(f"  Levels      : {[f'{v:.4f}' for v in QAM16_LEVELS.tolist()]}")
    print(f"  Architecture: QAM16ResNet  base_ch={BASE_CH}  blocks/stage={N_BLOCKS}")
    print(f"  Stages      : {BASE_CH}ch→{BASE_CH*2}ch→{BASE_CH*4}ch  (32→16→8 spatial)")
    print(f"  Quantized   : inputs + all internal weights + activations")
    print(f"  Float       : first conv + skip projections + classifier head (~1%)")
    print(f"  Epochs      : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Peak LR: {LR}")
    print(f"  Warmup      : {WARMUP} epochs  |  CutMix p={CUTMIX_PROB}")
    print(f"  Target      : 85%+ (requirement: 82%+)")
    print("=" * 72)

    # Data
    train_loader, test_loader = get_dataloaders(BATCH_SIZE)

    # Model
    model = QAM16ResNet(base_ch=BASE_CH, n_blocks=N_BLOCKS).to(device)
    total, fp, qp = model.param_summary()
    print(f"\nParameters : {total:,} total")
    print(f"  Float     : {fp:,}  ({100*fp/total:.1f}%)")
    print(f"  Quantized : {qp:,}  ({100*qp/total:.1f}%)")

    # Sanity-check input quantizer before any training
    sample_imgs, _ = next(iter(test_loader))
    print("\n[Input Quantizer Sanity Check]")
    model.input_q.verify(sample_imgs[:64].to(device))

    # ── Optimizer: separate weight decay for float vs scale/BN/bias params ──
    # BN params, biases, and learned scale alphas should NOT have weight decay:
    # BN/bias have their own dynamics; α needs freedom to grow/shrink.
    decay_params   = []
    nodecay_params = []
    for name, param in model.named_parameters():
        if ('bn' in name or name.endswith('.bias') or 'alpha' in name):
            nodecay_params.append(param)
        else:
            decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [{'params': decay_params,   'weight_decay': WEIGHT_DECAY},
         {'params': nodecay_params, 'weight_decay': 0.0}],
        lr=LR, betas=(0.9, 0.999), eps=1e-8
    )
    scheduler = make_scheduler(optimizer, WARMUP, EPOCHS)

    best_acc  = 0.0

    print(f"\nTraining...\n")

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer,
            epoch, WARMUP, CUTMIX_PROB,
            log_grads=(epoch == 1)
        )
        te_acc = evaluate(model, test_loader)
        scheduler.step()

        is_best = te_acc > best_acc
        if is_best:
            best_acc = te_acc
            torch.save(model.state_dict(), SAVE_NAME)

        lr_now = optimizer.param_groups[0]['lr']
        mark   = "  ★ NEW BEST" if is_best else ""
        print(f"Ep {epoch:3d}/{EPOCHS} | "
              f"Loss {tr_loss:.4f} | "
              f"Train {tr_acc:.2f}% | "
              f"Test {te_acc:.2f}% | "
              f"Best {best_acc:.2f}% | "
              f"LR {lr_now:.2e}"
              f"{mark}")

        # ── Per-layer diagnostics ────────────────────────────────────────
        if epoch % DIAG_EVERY == 0:
            print(f"\n  [DIAGNOSTICS — Epoch {epoch}]")
            for name, m in model.named_modules():
                if isinstance(m, QAM16Conv2d):
                    wn = torch.tanh(m.weight) * QAM16_MAX
                    wq = qam16_snap(wn, m.levels)
                    with torch.no_grad():
                        n_unique = wq.unique().numel()
                        snap_err = (wn - wq).abs().mean().item()
                        alpha_m  = m.alpha.abs().mean().item()
                    print(f"  {name:40s}  "
                          f"unique={n_unique}/4  "
                          f"snap_err={snap_err:.4f}  "
                          f"α={alpha_m:.3f}")
            print()

    # ── Final report ────────────────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"Training complete!")
    print(f"Best test accuracy : {best_acc:.2f}%")
    print(f"Target (82%+)      : {'✓  TARGET MET' if best_acc >= 82.0 else '✗  below target'}")
    print(f"Model saved to     : {SAVE_NAME}")
    print(f"{'='*72}")


if __name__ == "__main__":
    main()